In [ ]:
!pip install -q mwparserfromhell

In [ ]:
import json
import os
import time
import datetime
import requests
import mwparserfromhell

API_URL = "https://starwars.fandom.com/api.php"
HEADERS = {"User-Agent": "VaderDatasetBuilder/1.0 (personal research project)"}

OUT_DIR = "/kaggle/working/star_wars_raw"
RESUME_FROM_INPUT = None  # e.g. "/kaggle/input/star-wars-scrape-v3/star_wars_raw"

TITLES_PATH = os.path.join(OUT_DIR, "titles.json")
PROGRESS_PATH = os.path.join(OUT_DIR, "progress.json")
ARTICLES_PATH = os.path.join(OUT_DIR, "star_wars_articles.jsonl")

BATCH_SIZE = 20
FLUSH_EVERY = 100
MIN_TEXT_LEN = 200
SCRAPE_DATE = datetime.date.today().isoformat()

MAX_RETRIES = 5
RETRY_BACKOFF = 2.0


def api_get(params):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(API_URL, params=params, headers=HEADERS, timeout=30)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError) as e:
            if attempt == MAX_RETRIES:
                raise
            wait = RETRY_BACKOFF * (2 ** (attempt - 1))
            print(f"[retry] request failed ({e}), attempt {attempt}/{MAX_RETRIES}, waiting {wait:.0f}s")
            time.sleep(wait)


def bootstrap_from_previous_run():
    if RESUME_FROM_INPUT and not os.path.exists(OUT_DIR):
        if os.path.exists(RESUME_FROM_INPUT):
            import shutil
            shutil.copytree(RESUME_FROM_INPUT, OUT_DIR)
            print(f"[resumed] copied prior output forward from {RESUME_FROM_INPUT}")
        else:
            print(f"[warn] RESUME_FROM_INPUT set but not found at {RESUME_FROM_INPUT}, starting fresh")


def get_all_page_titles():
    if os.path.exists(TITLES_PATH):
        with open(TITLES_PATH) as f:
            titles = json.load(f)
        print(f"[cache] {len(titles):,} titles already fetched, skipping phase 1")
        return titles

    titles = []
    apcontinue = None
    while True:
        params = {
            "action": "query", "list": "allpages", "apnamespace": 0,
            "apfilterredir": "nonredirects",
            "aplimit": "500", "format": "json",
        }
        if apcontinue:
            params["apcontinue"] = apcontinue
        data = api_get(params)
        titles.extend(p["title"] for p in data["query"]["allpages"])
        if "continue" in data:
            apcontinue = data["continue"]["apcontinue"]
        else:
            break
        time.sleep(0.2)

    with open(TITLES_PATH, "w") as f:
        json.dump(titles, f)
    print(f"[done] {len(titles):,} titles fetched, cached to {TITLES_PATH}")
    return titles


def get_plaintext(titles_batch):
    params = {
        "action": "query", "prop": "revisions|info",
        "rvprop": "content", "rvslots": "main", "inprop": "url",
        "titles": "|".join(titles_batch), "format": "json",
    }
    data = api_get(params)
    pages = data.get("query", {}).get("pages", {})

    result = {}
    for p in pages.values():
        title = p.get("title", "")
        url = p.get("fullurl", "")
        revisions = p.get("revisions", [])
        wikitext = revisions[0].get("slots", {}).get("main", {}).get("*", "") if revisions else ""

        if wikitext.strip().upper().startswith("#REDIRECT"):
            continue  # belt-and-braces, apfilterredir should already exclude these

        text = mwparserfromhell.parse(wikitext).strip_code() if wikitext else ""
        result[title] = {"text": text, "url": url}
    return result


def load_progress():
    if os.path.exists(PROGRESS_PATH):
        with open(PROGRESS_PATH) as f:
            return json.load(f)
    return {"titles_done": 0, "articles_written": 0}


def save_progress(state):
    tmp = PROGRESS_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump(state, f)
    os.replace(tmp, PROGRESS_PATH)


os.makedirs(OUT_DIR, exist_ok=True)
print("functions loaded")

In [ ]:
sample = get_plaintext(["Luke Skywalker", "Anakin Skywalker", "Tatooine"])
ok = True
for title, page in sample.items():
    length = len(page["text"])
    print(f"{title}: {length} chars, url={page['url']}")
    if length < MIN_TEXT_LEN:
        ok = False

assert ok, "one or more sample pages returned no real text, do NOT run the scrape cell"
print("\n[passed] safe to run the scrape cell")

In [ ]:
bootstrap_from_previous_run()

titles = get_all_page_titles()
state = load_progress()

if state["titles_done"] > 0:
    print(f"[resumed] {state['titles_done']:,}/{len(titles):,} titles already processed, "
          f"{state['articles_written']:,} articles written so far")

buffer = []
with open(ARTICLES_PATH, "a", encoding="utf-8") as out_f:
    for i in range(state["titles_done"], len(titles), BATCH_SIZE):
        batch = titles[i:i + BATCH_SIZE]
        pages = get_plaintext(batch)

        for title, page in pages.items():
            if len(page["text"]) < MIN_TEXT_LEN:
                continue
            buffer.append(json.dumps({
                "title": title,
                "text": page["text"],
                "url": page["url"],
                "source": "Wookieepedia",
                "license": "CC BY-SA 3.0",
                "scrape_date": SCRAPE_DATE,
            }) + "\n")

        state["titles_done"] = min(i + BATCH_SIZE, len(titles))

        if len(buffer) >= FLUSH_EVERY:
            out_f.writelines(buffer)
            out_f.flush()
            state["articles_written"] += len(buffer)
            buffer = []
            save_progress(state)

        if state["titles_done"] % 2000 == 0:
            print(f"{state['titles_done']:,}/{len(titles):,} titles processed, "
                  f"{state['articles_written']:,} articles written")

        time.sleep(0.3)

    if buffer:
        out_f.writelines(buffer)
        state["articles_written"] += len(buffer)
        save_progress(state)

print(f"[complete] {state['articles_written']:,} articles in {ARTICLES_PATH}")

In [ ]:
import pandas as pd

if not os.path.exists(ARTICLES_PATH) or os.path.getsize(ARTICLES_PATH) == 0:
    raise SystemExit(
        f"[error] {ARTICLES_PATH} is missing or empty. "
        f"Has the scrape cell actually written any articles yet? Check progress.json."
    )

df = pd.read_json(ARTICLES_PATH, lines=True)

if "text" not in df.columns:
    raise SystemExit(
        f"[error] parsed {len(df)} rows but no 'text' column found, "
        f"the JSONL may be truncated or corrupted, check it manually."
    )

print(f"{len(df):,} articles, {df['text'].str.len().sum():,} characters total")

before = len(df)
df = df.drop_duplicates(subset="title", keep="last")
if len(df) != before:
    print(f"dropped {before - len(df):,} duplicate titles")

PARQUET_PATH = "/kaggle/working/star_wars_corpus.parquet"
df.to_parquet(PARQUET_PATH, index=False)
print(f"wrote {PARQUET_PATH}")